# <span style="color: #8ab0bc;">バージョン情報(Version Information)</span>

### V0.5

* 1) age      : 顧客の年齢。
* 2) job      : 職業（admin, blue-collar, technician など）。
* 3) marital  : 既婚・未婚などの婚姻状況。
* 4) education: 学歴（primary, secondary, tertiary など）
                    ※primary: 初等教育（小学校など）
                     secondary: 中等教育（中学校・高校など）
                     tertiary: 高等教育（大学・大学院など）
                     unknown: 不明
* 5) default  : 債務不履行があるかどうか（yes / no）　過去にローンやクレジットカードの支払いが滞ったことがあるか
* 6) balance  : 年間の平均残高。
* 7) housing  : 住宅ローンがあるかどうか（yes / no）。
* 8) loan     : 個人ローンがあるかどうか（yes / no）
* 9) contact  : 連絡手段（cellular（スマホ）, telephone （固定）など）。
* 10) day      : 最終接触日の「日」。
* 11) month    : 最終接触日の「月」。
* ※　duration削除
* 13) campaign : このキャンペーン（勧誘）期間中の接触回数。
* 14) pdays    : 前回のキャンペーン（勧誘）接触から経過した日数（-1は未接触）。
* 15) previous : このキャンペーン（勧誘）以前の接触回数。
* 16) poutcome : 前回のキャンペーン（勧誘）の成果（success(契約してくれた), failure(断られた) など）。
* 17) total_loan_count： housingとloanの「yes」の合計数（負債の重さ）。
* 18) balance_per_age： 年齢に対する残高の比率
* 19) is_new_prospect： previousが0かつpdaysが-1の完全新規フラグ。
* 20) day_month_interaction： 月と日の組み合わせによるボーナス・決算期特定

新しいカラム<br>

* 21) pdays_is_contacted： 過去に一度でも接触したことがあるか
* 22) balance_is_negative： 残高がマイナス（借金状態）かどうか
* 23) job_category_quality： 安定収入層（management, technician, admin.）判定
* 24) is_peak_campaign_month： キャンペーン集中月（5, 6, 7, 8月など）判定
* 25) age_bin_senior： 60歳以上の高齢層（退職金期待層）判定

# <span style="color: #8ab0bc;">ライブラリインポート(Library Imports)</span>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

# <span style="color: #8ab0bc;">関数定義(function definition)</span>

In [2]:
def df_Data_Cleansing(df):
    #削除データ
    df = df.drop(columns=['duration'])

    #追加データ
    df["total_loan_count"] = (df['housing'] == 'yes').astype(int) + (df['loan'] == 'yes').astype(int)  #17
    df['balance_per_age'] = (df['balance'] / df['age'] > 35.1).astype(int) #18
    df['is_new_prospect'] = ((df['previous'] == 0) & (df['pdays'] == -1)).astype(int) #19
    df["day_month_interaction"] = ((df['month'].isin(['mar', 'jun', 'sep', 'dec'])) & (df['day'] >= 20)).astype(int) #20
    
    #V0.5追加
    df['pdays_is_contacted'] = (df['pdays'] != -1).astype(int) #21
    df['balance_is_negative'] = (df['balance'] < 0).astype(int) #22
    df['job_category_quality'] = df['job'].isin(['management', 'technician', 'admin.']).astype(int) #23
    df['is_peak_campaign_month'] = df['month'].isin(['may', 'jun', 'jul', 'aug']).astype(int) #24
    df['age_bin_senior'] = (df['age'] >= 60).astype(int) #25

    return df

# <span style="color: #8ab0bc;">データ加工(Data Preprocessing)</span>

In [3]:
df = df_Data_Cleansing(pd.read_csv('data/train.csv')) 
df_test = df_Data_Cleansing(pd.read_csv('data/test.csv'))

df.head(5) 

,id,age,job,marital,education,default,balance,housing,loan,contact,...,y,total_loan_count,balance_per_age,is_new_prospect,day_month_interaction,pdays_is_contacted,balance_is_negative,job_category_quality,is_peak_campaign_month,age_bin_senior
0,0,42,technician,married,secondary,no,7,no,no,cellular,...,0,0,0,1,0,0,0,1,1,0
1,1,38,blue-collar,married,secondary,no,514,no,no,unknown,...,0,0,0,1,0,0,0,0,1,0
2,2,36,blue-collar,married,secondary,no,602,yes,no,unknown,...,0,1,0,1,0,0,0,0,1,0
3,3,27,student,single,secondary,no,34,yes,no,unknown,...,0,1,0,1,0,0,0,0,1,0
4,4,26,technician,married,secondary,no,889,yes,no,cellular,...,1,1,0,1,0,0,0,1,0,0


In [4]:
df.tail(5)

,id,age,job,marital,education,default,balance,housing,loan,contact,...,y,total_loan_count,balance_per_age,is_new_prospect,day_month_interaction,pdays_is_contacted,balance_is_negative,job_category_quality,is_peak_campaign_month,age_bin_senior
749995,749995,29,services,single,secondary,no,1282,no,yes,unknown,...,1,1,1,1,0,0,0,0,1,0
749996,749996,69,retired,divorced,tertiary,no,631,no,no,cellular,...,0,0,0,1,0,0,0,0,1,1
749997,749997,50,blue-collar,married,secondary,no,217,yes,no,cellular,...,0,1,0,1,0,0,0,0,0,0
749998,749998,32,technician,married,secondary,no,-274,no,no,cellular,...,0,0,0,1,0,0,1,1,1,0
749999,749999,42,technician,married,secondary,no,1559,no,no,cellular,...,0,0,1,0,0,1,0,1,1,0


In [5]:
df_test.head(5)

,id,age,job,marital,education,default,balance,housing,loan,contact,...,poutcome,total_loan_count,balance_per_age,is_new_prospect,day_month_interaction,pdays_is_contacted,balance_is_negative,job_category_quality,is_peak_campaign_month,age_bin_senior
0,750000,32,blue-collar,married,secondary,no,1397,yes,no,unknown,...,unknown,1,1,1,0,0,0,0,1,0
1,750001,44,management,married,tertiary,no,23,yes,no,cellular,...,unknown,1,0,1,0,0,0,1,0,0
2,750002,36,self-employed,married,primary,no,46,yes,yes,cellular,...,unknown,2,0,1,0,0,0,0,1,0
3,750003,58,blue-collar,married,secondary,no,-1380,yes,yes,unknown,...,unknown,2,0,1,0,0,1,0,1,0
4,750004,28,technician,single,secondary,no,1950,yes,no,cellular,...,unknown,1,1,1,0,0,0,1,1,0


In [6]:
df_test.tail(5)

,id,age,job,marital,education,default,balance,housing,loan,contact,...,poutcome,total_loan_count,balance_per_age,is_new_prospect,day_month_interaction,pdays_is_contacted,balance_is_negative,job_category_quality,is_peak_campaign_month,age_bin_senior
249995,999995,43,management,married,tertiary,no,0,yes,no,cellular,...,unknown,1,0,1,0,0,0,1,0,0
249996,999996,40,services,married,unknown,no,522,yes,no,cellular,...,failure,1,0,0,0,1,0,0,0,0
249997,999997,63,retired,married,primary,no,33,no,no,cellular,...,success,0,0,0,0,1,0,0,1,1
249998,999998,50,blue-collar,married,primary,no,2629,yes,no,unknown,...,unknown,1,1,1,0,0,0,0,1,0
249999,999999,29,student,single,tertiary,no,722,no,no,cellular,...,unknown,0,0,1,0,0,0,0,0,0


# <span style="color: #8ab0bc;">特徴量追加分(Feature add)</span>

## 17)  "total_loan_count"
[多重でローンしている人]<br>
0:借りていない<br>
1:個人ローン or 住宅ローンどちらかを借りている<br>
2:両方とも借りている <br>

In [7]:
df["total_loan_count"].value_counts()

total_loan_count
1    384545
0    299595
2     65860
Name: count, dtype: int64

In [8]:
df_test["total_loan_count"].value_counts()

total_loan_count
1    127659
0    100382
2     21959
Name: count, dtype: int64

## 18) "balance_per_age"
[年齢に対する残高の比率]<br>
1歳あたりいくら貯金しているか?
20歳の100万円と60歳の100万円を、モデルが同じ「100万円」として扱わないようにする<br>
※「35.1（上位25%）」より高ければ1、それ以外は0

In [9]:
df["balance_per_age"].describe() #四分位

count    750000.000000
mean          0.250255
std           0.433160
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max           1.000000
Name: balance_per_age, dtype: float64

In [10]:
df_test["balance_per_age"].describe() #四分位

count    250000.000000
mean          0.249812
std           0.432905
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: balance_per_age, dtype: float64

In [11]:
df["balance_per_age"].value_counts()

balance_per_age
0    562309
1    187691
Name: count, dtype: int64

In [12]:
df_test["balance_per_age"].value_counts()

balance_per_age
0    187547
1     62453
Name: count, dtype: int64

## 19) "is_new_prospect"
本当の意味での新規客
previous == 0：今回のキャンペーンより前に、一度も接触がない。<br>
かつ<br>
pdays == -1：過去に接触した記録（日数）が全く存在しない

In [13]:
df["is_new_prospect"].value_counts()

is_new_prospect
1    672417
0     77583
Name: count, dtype: int64

In [14]:
df_test["is_new_prospect"].value_counts()

is_new_prospect
1    224106
0     25894
Name: count, dtype: int64

## 20) "day_month_interaction"
[四半期末かつ20日以降]<br>

In [15]:
df["day_month_interaction"].value_counts()

day_month_interaction
0    729495
1     20505
Name: count, dtype: int64

In [16]:
df_test["day_month_interaction"].value_counts()

day_month_interaction
0    243042
1      6958
Name: count, dtype: int64

## 21) 'pdays_is_contacted'
過去に一度でも接触したことがあるかないか<br>
1:接触済み<br>
0:未接触<br>

In [17]:
df['pdays_is_contacted'].value_counts()

pdays_is_contacted
0    672434
1     77566
Name: count, dtype: int64

In [18]:
df_test['pdays_is_contacted'].value_counts()

pdays_is_contacted
0    224112
1     25888
Name: count, dtype: int64

## 22) 'balance_is_negative' <br>
残高がマイナスかどうか<br>
1:マイナス<br>
0:マイナスじゃない<br>

In [19]:
df["balance_is_negative"].value_counts()

balance_is_negative
0    645355
1    104645
Name: count, dtype: int64

In [20]:
df_test["balance_is_negative"].value_counts()

balance_is_negative
0    215266
1     34734
Name: count, dtype: int64

## 23) 'job_category_quality'
安定収入層かどうか？<br>
1:    jobが<br>
     'management'：（経営・管理職）,<br>
     'technician'：（技術職）,<br>
     'admin.'：（事務職）<br>
      の場合<br>
0:    それ以外<br>

In [21]:
df['job_category_quality'].value_counts()

job_category_quality
1    395140
0    354860
Name: count, dtype: int64

In [22]:
df_test['job_category_quality'].value_counts()

job_category_quality
1    131581
0    118419
Name: count, dtype: int64

## 24) 'is_peak_campaign_month'
キャンペーン集中月（5,6,7,8月）か<br>
1：はい<br>
0：いいえ<br>

In [23]:
df['is_peak_campaign_month'].value_counts()

is_peak_campaign_month
1    561587
0    188413
Name: count, dtype: int64

In [24]:
df_test['is_peak_campaign_month'].value_counts()

is_peak_campaign_month
1    186906
0     63094
Name: count, dtype: int64

## 25) "age_bin_senior"
60歳以上の高齢層か<br>
1：はい<br>
0：いいえ<br>

In [25]:
df['age_bin_senior'].value_counts()

age_bin_senior
0    726431
1     23569
Name: count, dtype: int64

In [26]:
df_test['age_bin_senior'].value_counts()

age_bin_senior
0    242244
1      7756
Name: count, dtype: int64

# <span style="color: #8ab0bc;">学習データ(traindata)</span>

In [27]:
df

,id,age,job,marital,education,default,balance,housing,loan,contact,...,y,total_loan_count,balance_per_age,is_new_prospect,day_month_interaction,pdays_is_contacted,balance_is_negative,job_category_quality,is_peak_campaign_month,age_bin_senior
0,0,42,technician,married,secondary,no,7,no,no,cellular,...,0,0,0,1,0,0,0,1,1,0
1,1,38,blue-collar,married,secondary,no,514,no,no,unknown,...,0,0,0,1,0,0,0,0,1,0
2,2,36,blue-collar,married,secondary,no,602,yes,no,unknown,...,0,1,0,1,0,0,0,0,1,0
3,3,27,student,single,secondary,no,34,yes,no,unknown,...,0,1,0,1,0,0,0,0,1,0
4,4,26,technician,married,secondary,no,889,yes,no,cellular,...,1,1,0,1,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
749995,749995,29,services,single,secondary,no,1282,no,yes,unknown,...,1,1,1,1,0,0,0,0,1,0
749996,749996,69,retired,divorced,tertiary,no,631,no,no,cellular,...,0,0,0,1,0,0,0,0,1,1
749997,749997,50,blue-collar,married,secondary,no,217,yes,no,cellular,...,0,1,0,1,0,0,0,0,0,0
749998,749998,32,technician,married,secondary,no,-274,no,no,cellular,...,0,0,0,1,0,0,1,1,1,0


# <span style="color: #8ab0bc;">試験データ(test data)</span>

In [28]:
df_test

,id,age,job,marital,education,default,balance,housing,loan,contact,...,poutcome,total_loan_count,balance_per_age,is_new_prospect,day_month_interaction,pdays_is_contacted,balance_is_negative,job_category_quality,is_peak_campaign_month,age_bin_senior
0,750000,32,blue-collar,married,secondary,no,1397,yes,no,unknown,...,unknown,1,1,1,0,0,0,0,1,0
1,750001,44,management,married,tertiary,no,23,yes,no,cellular,...,unknown,1,0,1,0,0,0,1,0,0
2,750002,36,self-employed,married,primary,no,46,yes,yes,cellular,...,unknown,2,0,1,0,0,0,0,1,0
3,750003,58,blue-collar,married,secondary,no,-1380,yes,yes,unknown,...,unknown,2,0,1,0,0,1,0,1,0
4,750004,28,technician,single,secondary,no,1950,yes,no,cellular,...,unknown,1,1,1,0,0,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,999995,43,management,married,tertiary,no,0,yes,no,cellular,...,unknown,1,0,1,0,0,0,1,0,0
249996,999996,40,services,married,unknown,no,522,yes,no,cellular,...,failure,1,0,0,0,1,0,0,0,0
249997,999997,63,retired,married,primary,no,33,no,no,cellular,...,success,0,0,0,0,1,0,0,1,1
249998,999998,50,blue-collar,married,primary,no,2629,yes,no,unknown,...,unknown,1,1,1,0,0,0,0,1,0


# <span style="color: #8ab0bc;">学習モデル作成(Model Training)</span>

In [29]:
y = df["y"]
x = df.drop(columns=["y", "id"])
x = pd.get_dummies(x, drop_first=True)


In [30]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42)

In [31]:
train_data = lgb.Dataset(x_train,label=y_train)
params = {
    'objective': 'binary',
    'boosting_type': 'gbdt',
    'metric': 'binary_logloss',
    'num_leaves': 31,
    'learning_rate': 0.05    
}

In [32]:
model = lgb.train(params, train_data,num_boost_round=100)

[LightGBM] [Info] Number of positive: 72283, number of negative: 527717
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017487 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 790
[LightGBM] [Info] Number of data points in the train set: 600000, number of used features: 50
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.120472 -> initscore=-1.987971
[LightGBM] [Info] Start training from score -1.987971


In [33]:
y_pred_proba = model.predict(x_test)
y_pred_proba

array([0.11080009, 0.66478079, 0.05244986, ..., 0.10441218, 0.11384926,
       0.28307874], shape=(150000,))

In [34]:
auc = roc_auc_score(y_test, y_pred_proba)
print("roc-auc : ",auc)

roc-auc :  0.8421719373317439


# <span style="color: #8ab0bc;">テスト、提出物生成(Inference & Submission Generation)</span>

In [35]:
submit_x = df_test.drop(columns=["id"])
submit_x = pd.get_dummies(submit_x, drop_first=True)
submit_x = submit_x.reindex(columns=x.columns, fill_value=0)

In [36]:
submit_x

,age,balance,day,campaign,pdays,previous,total_loan_count,balance_per_age,is_new_prospect,day_month_interaction,...,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success,poutcome_unknown
0,32,1397,21,1,-1,0,1,1,1,0,...,False,False,False,True,False,False,False,False,False,True
1,44,23,3,2,-1,0,1,0,1,0,...,False,False,False,False,False,False,False,False,False,True
2,36,46,13,2,-1,0,2,0,1,0,...,False,False,False,True,False,False,False,False,False,True
3,58,-1380,29,1,-1,0,2,0,1,0,...,False,False,False,True,False,False,False,False,False,True
4,28,1950,22,1,-1,0,1,1,1,0,...,True,False,False,False,False,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,43,0,18,2,-1,0,1,0,1,0,...,False,False,False,False,True,False,False,False,False,True
249996,40,522,19,1,189,1,1,0,0,0,...,False,False,False,False,True,False,False,False,False,False
249997,63,33,3,1,92,8,0,0,0,0,...,True,False,False,False,False,False,False,False,True,False
249998,50,2629,30,2,-1,0,1,1,1,0,...,False,False,False,True,False,False,False,False,False,True


In [37]:
test_pred = model.predict(submit_x)
test_pred

array([0.05551861, 0.06533403, 0.06458787, ..., 0.82791991, 0.04969335,
       0.42949175], shape=(250000,))

In [42]:
submission = pd.DataFrame({
    'id': df_test['id'],
    'y': test_pred
})
submission.to_csv('submission.csv', index=False)
submit_x.to_csv("submit_x_x_x_x_x_x.csv", index=False)